In [1]:
from scipy.stats import norm
import biogeme.biogeme as bio
from biogeme.expressions import Beta, Variable, bioDraws, MonteCarlo, exp, log, Elem, bioNormalCdf
from biogeme import models
from biogeme.models import ordered_probit, ordered_logit
from biogeme import results as res
from biogeme.results import compile_estimation_results, calcPValue
import pandas as pd
import biogeme.database as db
import numpy as np
import biogeme.distributions as dist
import pickle
from urllib.request import urlopen
import os

In [2]:
df=pd.read_csv('final_processed_crash_dataset.csv')

/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_22598/3930622460.py:1: DtypeWarning: Columns (11,17,51,52,54,56,57,58,60,61,66,67,68,69,70,71,73,74,76,77,78,84,86,87,88,99,100,101,102,106,110,113,114,115,116,117,118,119,120,121,122,124,133) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('final_processed_crash_dataset.csv')


## Sanity check and preparing the databases

In [3]:
df=df[df['severity']!=-1]
df=df.loc[df['Vehicle'].isin(['E-PMD','Bike','E-bike','Pedestrian'])]

df_dummies=pd.get_dummies(df[['Crossroad','Helmet','Point of impact','Gender','vehicle_type_2','Vehicle','Pavement','Intersection'
                              , 'User category', 'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration', 'Age category',
                              'Accident location', 'Surface condition', 'Maneuver', 'Maneuver_2','Gender_2', 'Pedestrian localisation', 'Pedestrian action', 'Vehicle_2',
'Max speed', 'Long profile', 'Weather conditions', 'Road type', 'Trip purpose','Reflective jacket', 'plan','Point of impact_2', 'Obstacle','Gender_driver','Helmet_driver','age_driver','Year', 'Maneuver_3','Point of impact_3','Gender_3','Vehicle_3'

]])
df_dummies = df_dummies.astype(int)  # Convert boolean to integers

# Select columns that are not of type object
df_non_dummies = df[['age','severity','Number of passengers','number of involved vehicles','vma','age_2','catu', 'Num_Acc','age_opposite_mean']]



In [4]:
df_non_dummies=pd.concat([df_non_dummies,df_dummies],axis=1)

## Removing useless rows
df_non_dummies=df_non_dummies.dropna(subset=['age','severity'])


df_non_dummies['age_opposite_mean'] = df_non_dummies['age_opposite_mean'].fillna(999)

df_non_dummies = df_non_dummies.reset_index(drop=True)

In [5]:
## First model
df_carcrashes=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Cars']==1) | (df_non_dummies['vehicle_type_2_Large motorized vehicle'] ==1) | (df_non_dummies['vehicle_type_2_Light motorized vehicle']==1) ]
df_carcrashes=df_carcrashes.loc[df_carcrashes['catu'].isin([1,2])]
database_carcrashes = db.Database('database_carcrashes', df_carcrashes)

## Second model
df_mmv=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Micromobility vehicle']==1) ]
df_mmv=df_mmv.loc[df_mmv['catu'].isin([1,2])]
df_mmv=df_mmv.loc[df_mmv['severity']!=3] ## Only one value
df_mmv = df_mmv.loc[df_mmv['age_2'] != 999].copy()

database_mmv= db.Database('database_mmv',df_mmv)


## Third model
num_acc_values = df_non_dummies.loc[df_non_dummies['vehicle_type_2_Pedestrian'] == 1, 'Num_Acc']
df_pedestrian = df_non_dummies[df_non_dummies['Num_Acc'].isin(num_acc_values)]
df_pedestrian = df_pedestrian.loc[df_pedestrian['age_2'] != 999].copy()

database_pedestrian= db.Database('database_pedestrian',df_pedestrian)

## Fourth model
df_sv=df_non_dummies.loc[df_non_dummies['vehicle_type_2_No other vehicle']==1]
df_sv=df_sv.loc[df_sv['catu'].isin([1,2])]
database_sv= db.Database('database_sv',df_sv)


## Variables and Betas

In [6]:
import re
import biogeme.database as db
from biogeme.expressions import Beta, Variable


def normalize(name: str) -> str:
    clean = name.lower()
    clean = re.sub(r'[^a-z0-9]+', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    clean = clean.strip('_')
    if re.match(r'^[0-9]', clean):
        clean = "var_" + clean
    return clean

base_suffixes = ['','_I', '_F', '_mean', '_I_mean', '_F_mean','_sd','_I_std','_F_std']
modes = ['', '_epmd', '_bike', '_ebike']  

for col in df_non_dummies.columns:
    clean_name = normalize(col)

    # Skip si nettoyage donne v_injuryde
    if clean_name == "":
        print(f"Skip (empty after cleaning): {col}")
        continue

    if clean_name not in locals():
        exec(f"{clean_name} = Variable({repr(col)})")
    # Pour chaque suffixe de base et chaque mode, créer un Beta si non existant
    for suf in base_suffixes:
        for mode in modes:
            beta_var_name = f"beta_{clean_name}{suf}{mode}"
            beta_label = beta_var_name

            if '_std' in suf or suf == '_std':
                init_value = 1
            else:
                init_value = 0

            if not beta_var_name.isidentifier():
                print(f"Skip beta (invalid identifier): {beta_var_name}")
                continue

            if beta_var_name in locals():
                continue
            exec(f"{beta_var_name} = Beta({repr(beta_label)}, {init_value}, None, None, 0)")

constant_I = Beta('constant_I', 0, None, None, 0)
constant_F = Beta('constant_F', 0, None, None, 0)


## Model for car crashes

In [7]:
v_no_injury = 0

v_injury = (  beta_gender_female_I * gender_female  + constant_I
      + beta_user_category_passenger_I * user_category_passenger
      + beta_number_of_involved_vehicles_I * number_of_involved_vehicles
      + beta_point_of_impact_back_I_bike * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
      + beta_point_of_impact_back_I_epmd* point_of_impact_back * vehicle_e_pmd
      + beta_vehicle_type_2_light_motorized_vehicle_I* vehicle_type_2_light_motorized_vehicle
      + beta_maneuver_2_overtaking_I * maneuver_2_overtaking
      + beta_maneuver_2_without_change_of_direction_I_bike * maneuver_without_change_of_direction * vehicle_bike
      + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
      + beta_intersection_no_intersection_I * intersection_no_intersection
)

v_fatality = (  constant_F
        + beta_user_category_passenger_I * user_category_passenger
        + beta_age_F * age
        + beta_vma_F * vma * intersection_no_intersection
        + beta_vehicle_type_2_large_motorized_vehicle_F * vehicle_type_2_large_motorized_vehicle
        +beta_vehicle_type_2_light_motorized_vehicle_F* vehicle_type_2_light_motorized_vehicle
        + beta_lighting_conditions_daylight_F * lighting_conditions_daylight
        + beta_maneuver_2_turning_left_F * maneuver_2_turning_left
        + beta_accident_location_on_cycle_facility_F* accident_location_on_cycle_facility
        + beta_lighting_conditions_night_with_street_lightings_on_F* lighting_conditions_night_with_street_lightings_on
        + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
)

utility_motorized_vehicles = {
    1: v_no_injury,
    2: v_injury,
    3: v_fatality
}


In [8]:
## Constant model

availability = {
    1: 1, 
    2: 1, 
    3: 1   
}

U={1:0,2:constant_I,3:constant_F}

model_name = 'InitialModel_carcrashes'


logprob = models.loglogit(U, availability, severity)

# Créez l'objet Biogeme
model_cst_car = bio.BIOGEME(database_carcrashes, logprob)
model_cst_car.modelName = model_name

# Estimation


results_constant_car = model_cst_car.estimate()



In [9]:
# Random-parameters model
sigma_I = Beta('sigma I', 0, None, None, 0)

X1 = bioDraws('X1', 'NORMAL')


# Adding the error component to the utilities
v_no_injury_rp=0
v_injury_rp = utility_motorized_vehicles[2] + sigma_I*X1
v_fatality_rp = utility_motorized_vehicles[3]

utility_motorized_vehicles_mixed={1:v_no_injury_rp,2:v_injury_rp,3:v_fatality_rp}

prob = models.logit(utility_motorized_vehicles_mixed,availability,severity)


# We integrate over B_TIME_RND using Monte-Carlo
logprob = log(MonteCarlo(prob))



# Create the Biogeme object
model_car  = bio.BIOGEME(database_carcrashes,logprob,number_of_draws=1)
model_car.modelName = "random_parameter_logit_car_crashes"

# Estimate the parameters. 
results_ml_carcrashes = model_car.estimate()


The number of draws (1) is low. The results may not be meaningful.


In [10]:
results_ml_carcrashes.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_accident_location_on_cycle_facility_F,-0.721633,0.501798,-1.438095,1.504072e-01
beta_age_F,0.036946,0.008803,4.197207,2.702266e-05
beta_gender_driver_female_I,-0.862326,0.418187,-2.062057,3.920231e-02
beta_gender_female_I,0.666215,0.137794,4.834866,1.332354e-06
beta_intersection_no_intersection_I,0.395891,0.143393,2.760885,5.764490e-03
beta_lighting_conditions_daylight_F,-1.045978,0.398197,-2.626788,8.619502e-03
beta_lighting_conditions_night_with_street_lightings_on_F,-0.314059,0.455892,-0.688889,4.908934e-01
beta_maneuver_2_overtaking_I,-0.384106,0.212651,-1.806269,7.087639e-02
beta_maneuver_2_turning_left_F,-0.995250,0.619929,-1.605426,1.084000e-01
beta_maneuver_2_without_change_of_direction_I_bike,0.509951,0.136482,3.736395,1.866774e-04


## MMV

In [11]:
v_no_injury = 0

v_injury = (
      beta_gender_female_I                  * gender_female
    + constant_I
    + beta_gender_2_female_I               * (gender_2_female + gender_3_male)
    + beta_point_of_impact_back_I_bike     * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
    + beta_point_of_impact_back_I_epmd     * point_of_impact_back * vehicle_e_pmd
    + beta_surface_condition_wet_I         * surface_condition_wet
    + beta_age_I                           * age
     + beta_maneuver_swerving_I           * maneuver_swerving
    + beta_maneuver_turning_left_I         * maneuver_turning_left
    + beta_age_2_I                          * age_2*(age_2 !=999)
    + beta_vehicle_2_e_pmd_I           * (vehicle_2_e_pmd + vehicle_3_e_pmd)
)

# Dictionnaire des fonctions d'utilité
utility_mmv = {
    1: v_no_injury,
    2: v_injury,
}


In [12]:
availability1={1:1,2:1}



logprob_2 = models.loglogit(utility_mmv, availability1, severity)
model_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_mmv.modelName = 'logit_mmv'
results_logit_mmv= model_mmv.estimate()
results_logit_mmv.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_2_I,-0.028348,0.004871,-5.820330,5.873152e-09
beta_age_I,0.033316,0.005233,6.366907,1.928775e-10
beta_gender_2_female_I,-0.904217,0.141067,-6.409832,1.456799e-10
beta_gender_female_I,1.050717,0.168342,6.241566,4.332106e-10
beta_maneuver_swerving_I,-0.501427,0.191476,-2.618740,8.825520e-03
beta_maneuver_turning_left_I,-1.282928,0.324463,-3.954011,7.685189e-05
beta_point_of_impact_back_I_bike,-1.093429,0.224785,-4.864339,1.148399e-06
beta_point_of_impact_back_I_epmd,1.144355,0.686199,1.667673,9.538063e-02
beta_surface_condition_wet_I,-0.596597,0.230741,-2.585568,9.721869e-03
beta_vehicle_2_e_pmd_I,-0.323717,0.171245,-1.890374,5.870798e-02


In [13]:
v_no_injurym=0
v_injurym = constant_I


U={1:v_no_injurym,2:v_injurym}

In [14]:
model_name = 'InitialModel_mmv'



logprob_2 = models.loglogit(U, availability1, severity)

# Créez l'objet Biogeme
model_cst_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_cst_mmv.modelName = model_name

# Estimation


results_constant_mmv = model_cst_mmv.estimate()



## Pedestrian

In [15]:
Beta_age_bike_4=Beta('Beta_age_bike_4',0,None,None,0)

In [16]:
utility_pedestrian = (
      beta_gender_female                     * gender_female
    + beta_age                               * age
    + beta_intersection_no_intersection      * intersection_no_intersection
    + beta_age_2                             * age_2
                                            * (number_of_involved_vehicles == 2)
    + beta_gender_2_female                   * gender_2_female
                                            * (number_of_involved_vehicles == 2)
    + beta_user_category_pedestrian          * user_category_pedestrian
    + beta_maneuver_2_turning_left           * user_category_pedestrian
                                            * (maneuver_2_turning_left + maneuver_2_turning_right)
                                            * (number_of_involved_vehicles == 2)
    + beta_crossroad_traffic_lights          * crossroad_traffic_lights
)


In [17]:
model_name = 'ordered_probit_pedestrian'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_pedestrian,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_pedes.modelName = model_name
results_pedes = model_pedes.estimate()
results_pedes.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.014941,0.001611,9.276570,0.000000e+00
beta_age_2,-0.007241,0.001451,-4.991082,6.004200e-07
beta_crossroad_traffic_lights,0.200818,0.077781,2.581848,9.827280e-03
beta_gender_2_female,-0.565388,0.065134,-8.680374,0.000000e+00
beta_gender_female,0.515462,0.066945,7.699805,1.354472e-14
beta_intersection_no_intersection,0.236975,0.069503,3.409555,6.506890e-04
beta_maneuver_2_turning_left,0.525749,0.111200,4.727952,2.267959e-06
beta_user_category_pedestrian,1.350926,0.071959,18.773448,0.000000e+00
tau_1,0.793637,0.106255,7.469199,8.060219e-14
tau_1_diff_2,4.169748,0.170605,24.440997,0.000000e+00


In [18]:
model_name = 'ordinal_probit_pedestrian_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes.modelName = model_name
results_pedes_cst = model_cst_pedes.estimate()


## Single-vehicle

In [19]:
beta_user_category_passenger_mixed=beta_user_category_passenger_mean + beta_user_category_passenger_sd*X1 * bioDraws('X1', 'NORMAL')


In [20]:
utility_sv = (
    beta_age * age  +
    beta_user_category_passenger_mixed * user_category_passenger 
+ beta_long_profile_slope *long_profile_slope 
+beta_helmet_driver_yes_ebike*helmet_driver_yes*vehicle_e_bike
+ beta_number_of_passengers*user_category_driver*(number_of_passengers==1) 
)


In [21]:
model_name = 'ordered_probit_sinv'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log(MonteCarlo(the_chosen_proba))
model_solo_2 = bio.BIOGEME(database_sv, logprob,number_of_draws=1)
model_solo_2.modelName = model_name
results_solo_2 = model_solo_2.estimate()
results_solo_2.get_estimated_parameters()

The number of draws (1) is low. The results may not be meaningful.


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.008455,0.003636,2.325738,2.003255e-02
beta_helmet_driver_yes_ebike,0.414934,0.704286,0.589156,5.557565e-01
beta_long_profile_slope,0.325841,0.167649,1.943588,5.194513e-02
beta_number_of_passengers,-2.236651,0.233696,-9.570786,0.000000e+00
beta_user_category_passenger_mean,-2.025237,0.330443,-6.128856,8.851340e-10
beta_user_category_passenger_sd,0.069193,0.095363,0.725573,4.681009e-01
tau_1,-2.516339,0.193022,-13.036517,0.000000e+00
tau_1_diff_2,5.188487,0.198280,26.167472,0.000000e+00


In [22]:
model_name = 'ordered_logit_s_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_solo = bio.BIOGEME(database_sv, logprob)
model_cst_solo.modelName = model_name
results_cst_solo = model_cst_solo.estimate()
results_cst_solo.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
tau_1,-2.115696,0.064929,-32.584718,0.0
tau_1_diff_2,4.395508,0.099081,44.362708,0.0


## Likelihood-ratio test

In [23]:
def get_results(file_path):

    # Ouvrir le fichier en mode binaire
    with open(file_path, 'rb') as file:
        data = pickle.load(file)

    result = res.bioResults(data)
    
    # Retourner le résultat
    return result


#res_restricted=get_results('logit_mmv~51.pickle')
#res_unrestricted=get_results('panel_mmv~36.pickle')

#res_restricted.likelihood_ratio_test(res_unrestricted, 0.01)

## Out-of sample validation of the models

In [24]:
# Create DataFrames for each year and without each year
years = [2019, 2020, 2021, 2022, 2023]

# Classe pour contenir les données de validation
class ValidationData:
    def __init__(self, estimation, validation):
        self.estimation = estimation
        self.validation = validation

def create_validation_data(df):
    validation_data=[]
    validation_data.append(ValidationData(df[df['Year'].isin([2019, 2020, 2021, 2022])], df[df['Year'] == 2023]))
    df_lyon = df[df['Agglomeration_MÉTROPOLE DE LYON'] == 1]
    df_paris = df[df['Agglomeration_MÉTROPOLE DU GRAND PARIS']==1]
    validation_data.append(ValidationData(df_paris, df_lyon))
    return validation_data



# Create validation data for each dataset
validationData_car = create_validation_data(df_carcrashes)
validationData_mmv= create_validation_data(df_mmv)
validationData_sv_2 = create_validation_data(df_sv)
validationData_pedestrian = create_validation_data(df_pedestrian)

In [25]:
# Validate the model with the validation data for mmv
validation_results_car = model_car.validate(results_ml_carcrashes, validationData_car)
validation_results_car_cst = model_cst_car.validate(results_ml_carcrashes, validationData_car)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_car):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_car_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_car)):
    validation_loglike = validation_results_car[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_car_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on car (slide {i+1}): {rho_square}')





KeyboardInterrupt: 

In [ ]:
# Validate the model with the validation data for mmv
validation_results_mmv = model_mmv.validate(results_logit_mmv, validationData_mmv)
validation_results_mmv_cst = model_cst_mmv.validate(results_constant_mmv, validationData_mmv)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')





Log likelihood for 374 validation data on mmv (slide 1): -192.33769159646437
Log likelihood for 87 validation data on mmv (slide 2): -41.86523242443988
Log likelihood for 374 validation data on mmv (constant model, slide 1): -237.10399908707495
Log likelihood for 87 validation data on mmv (constant model, slide 2): -53.45166827834558
Rho-square for the validation data on mmv (slide 1): 0.18880452317537855
Rho-square for the validation data on mmv (slide 2): 0.216764718990064


In [ ]:

# Validate the model with the validation data for mmv
validation_results_mmv = model_pedes.validate(results_pedes, validationData_pedestrian)
validation_results_mmv_cst = model_cst_pedes.validate(results_pedes_cst, validationData_pedestrian)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')




Log likelihood for 711 validation data on mmv (slide 1): -305.8780188215393
Log likelihood for 223 validation data on mmv (slide 2): -91.82829391930525
Log likelihood for 711 validation data on mmv (constant model, slide 1): -491.2867169242943
Log likelihood for 223 validation data on mmv (constant model, slide 2): -151.9994039479823
Rho-square for the validation data on mmv (slide 1): 0.3773940790899215
Rho-square for the validation data on mmv (slide 2): 0.39586411831765467


In [ ]:


# Validate the model with the validation data for mmv
validation_results_solo_2 = model_solo_2.validate(results_solo_2, validationData_sv_2)
validation_results_solo_cst = model_cst_solo.validate(results_cst_solo, validationData_sv_2)

# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_solo_2):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_solo_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)

    # Calculate rho-square for each slide (mmv
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')


KeyboardInterrupt: 